In [1]:
import json
import pandas as pd
from pathlib import Path

In [3]:
queries = pd.read_json("../../data/processed/retrieval/query_intents.jsonl", lines=True)
candidates = pd.read_parquet("../../data/processed/retrieval/candidate_sites.parquet")

v1_top50 = pd.read_parquet("../artifacts/models/two_tower_v1/evaluation/top50_retrieval.parquet")
v2_top50 = pd.read_parquet("../artifacts/models/two_tower_v2/evaluation/top50_retrieval.parquet")

print(len(queries), len(candidates), len(v1_top50), len(v2_top50))

21 50000 1050 1050


In [4]:
def score_col_for_strategy(strategy: str) -> str:
    return f"{strategy}_score"

def evaluate_retrieval_df(retrieval_df: pd.DataFrame, queries_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    match_rows = []
    score_rows = []

    for _, q in queries_df.iterrows():
        qid = q["query_id"]
        strategy = q["strategy"]
        score_col = score_col_for_strategy(strategy)

        qdf = retrieval_df[retrieval_df["query_id"] == qid].copy()
        top10 = qdf.head(10)
        top20 = qdf.head(20)

        match_rows.append({
            "query_id": qid,
            "strategy": strategy,
            "top10_match_rate": (top10["top_strategy"] == strategy).mean(),
            "top20_match_rate": (top20["top_strategy"] == strategy).mean(),
        })

        score_rows.append({
            "query_id": qid,
            "strategy": strategy,
            "top20_mean_score": top20[score_col].mean(),
            "top20_median_score": top20[score_col].median(),
            "top20_high_score_rate": (top20[score_col] >= 70).mean(),
        })

    return pd.DataFrame(match_rows), pd.DataFrame(score_rows)

def summarize_metrics(match_df: pd.DataFrame, score_df: pd.DataFrame) -> dict:
    return {
        "top10_match_rate_mean": float(match_df["top10_match_rate"].mean()),
        "top20_match_rate_mean": float(match_df["top20_match_rate"].mean()),
        "top20_mean_score_mean": float(score_df["top20_mean_score"].mean()),
        "top20_median_score_mean": float(score_df["top20_median_score"].mean()),
        "top20_high_score_rate_mean": float(score_df["top20_high_score_rate"].mean()),
    }

In [5]:
heuristic_rows = []

for _, q in queries.iterrows():
    qid = q["query_id"]
    strategy = q["strategy"]
    score_col = score_col_for_strategy(strategy)

    topk = candidates.sort_values(score_col, ascending=False).head(50).copy()
    topk["query_id"] = qid
    topk["query_text"] = q["text"]
    topk["strategy"] = strategy
    topk["similarity"] = None
    heuristic_rows.append(topk)

heuristic_top50 = pd.concat(heuristic_rows, ignore_index=True)
len(heuristic_top50)

1050

In [6]:
heur_match, heur_score = evaluate_retrieval_df(heuristic_top50, queries)
v1_match, v1_score = evaluate_retrieval_df(v1_top50, queries)
v2_match, v2_score = evaluate_retrieval_df(v2_top50, queries)

summary_df = pd.DataFrame(
    [
        summarize_metrics(heur_match, heur_score),
        summarize_metrics(v1_match, v1_score),
        summarize_metrics(v2_match, v2_score),
    ],
    index=["heuristic", "two_tower_v1", "two_tower_v2"]
)

summary_df

,top10_match_rate_mean,top20_match_rate_mean,top20_mean_score_mean,top20_median_score_mean,top20_high_score_rate_mean
heuristic,0.385714,0.421429,88.714286,88.714286,1.000000
two_tower_v1,0.438095,0.457143,61.870000,62.116667,0.492857
two_tower_v2,0.204762,0.185714,68.671429,68.714286,0.473810


In [7]:
V1_STRATEGIES = {
    "single_dwelling_rebuild",
    "granny_flat",
    "dual_occupancy",
    "land_bank_hold",
}

V2_STRATEGIES = {
    "assembly_opportunity",
    "townhouse_multi_dwelling",
    "low_rise_apartment",
}

In [8]:
routed_rows = []

for _, q in queries.iterrows():
    strategy = q["strategy"]
    qid = q["query_id"]

    if strategy in V1_STRATEGIES:
        source = v1_top50
    elif strategy in V2_STRATEGIES:
        source = v2_top50
    else:
        raise ValueError(strategy)

    qdf = source[source["query_id"] == qid].copy()
    routed_rows.append(qdf)

routed_top50 = pd.concat(routed_rows, ignore_index=True)
len(routed_top50)

1050

In [9]:
routed_match, routed_score = evaluate_retrieval_df(routed_top50, queries)

summary_df = pd.DataFrame(
    [
        summarize_metrics(heur_match, heur_score),
        summarize_metrics(v1_match, v1_score),
        summarize_metrics(v2_match, v2_score),
        summarize_metrics(routed_match, routed_score),
    ],
    index=["heuristic", "two_tower_v1", "two_tower_v2", "routed"]
)

summary_df

,top10_match_rate_mean,top20_match_rate_mean,top20_mean_score_mean,top20_median_score_mean,top20_high_score_rate_mean
heuristic,0.385714,0.421429,88.714286,88.714286,1.000000
two_tower_v1,0.438095,0.457143,61.870000,62.116667,0.492857
two_tower_v2,0.204762,0.185714,68.671429,68.714286,0.473810
routed,0.466667,0.495238,77.624048,77.714286,0.735714


In [10]:
def minmax_norm(s: pd.Series) -> pd.Series:
    if s.isna().all():
        return pd.Series([0.0] * len(s), index=s.index)
    s_min = s.min()
    s_max = s.max()
    if s_max == s_min:
        return pd.Series([0.0] * len(s), index=s.index)
    return (s - s_min) / (s_max - s_min)

def fuse_query_df(qdf: pd.DataFrame, strategy: str, alpha: float = 0.5, beta: float = 0.5) -> pd.DataFrame:
    score_col = score_col_for_strategy(strategy)
    out = qdf.copy()

    out["sim_norm"] = minmax_norm(out["similarity"].astype(float))
    out["score_norm"] = minmax_norm(out[score_col].astype(float))
    out["fusion_score"] = alpha * out["sim_norm"] + beta * out["score_norm"]

    return out.sort_values("fusion_score", ascending=False)

In [11]:
fusion_v1_rows = []

for _, q in queries.iterrows():
    qid = q["query_id"]
    strategy = q["strategy"]

    qdf = v1_top50[v1_top50["query_id"] == qid].copy()
    fused = fuse_query_df(qdf, strategy=strategy, alpha=0.5, beta=0.5).head(50)
    fusion_v1_rows.append(fused)

fusion_v1_top50 = pd.concat(fusion_v1_rows, ignore_index=True)

In [12]:
fusion_v1_match, fusion_v1_score = evaluate_retrieval_df(fusion_v1_top50, queries)

summary_df = pd.DataFrame(
    [
        summarize_metrics(heur_match, heur_score),
        summarize_metrics(v1_match, v1_score),
        summarize_metrics(v2_match, v2_score),
        summarize_metrics(routed_match, routed_score),
        summarize_metrics(fusion_v1_match, fusion_v1_score),
    ],
    index=["heuristic", "two_tower_v1", "two_tower_v2", "routed", "fusion_v1"]
)

summary_df

,top10_match_rate_mean,top20_match_rate_mean,top20_mean_score_mean,top20_median_score_mean,top20_high_score_rate_mean
heuristic,0.385714,0.421429,88.714286,88.714286,1.000000
two_tower_v1,0.438095,0.457143,61.870000,62.116667,0.492857
two_tower_v2,0.204762,0.185714,68.671429,68.714286,0.473810
routed,0.466667,0.495238,77.624048,77.714286,0.735714
fusion_v1,0.495238,0.504762,69.683810,69.428571,0.590476


In [13]:
fusion_v2_rows = []

for _, q in queries.iterrows():
    qid = q["query_id"]
    strategy = q["strategy"]

    qdf = v2_top50[v2_top50["query_id"] == qid].copy()
    fused = fuse_query_df(qdf, strategy=strategy, alpha=0.5, beta=0.5).head(50)
    fusion_v2_rows.append(fused)

fusion_v2_top50 = pd.concat(fusion_v2_rows, ignore_index=True)
fusion_v2_match, fusion_v2_score = evaluate_retrieval_df(fusion_v2_top50, queries)

In [14]:
summary_df = pd.DataFrame(
    [
        summarize_metrics(heur_match, heur_score),
        summarize_metrics(v1_match, v1_score),
        summarize_metrics(v2_match, v2_score),
        summarize_metrics(routed_match, routed_score),
        summarize_metrics(fusion_v1_match, fusion_v1_score),
        summarize_metrics(fusion_v2_match, fusion_v2_score),
    ],
    index=["heuristic", "two_tower_v1", "two_tower_v2", "routed", "fusion_v1", "fusion_v2"]
)

summary_df

,top10_match_rate_mean,top20_match_rate_mean,top20_mean_score_mean,top20_median_score_mean,top20_high_score_rate_mean
heuristic,0.385714,0.421429,88.714286,88.714286,1.000000
two_tower_v1,0.438095,0.457143,61.870000,62.116667,0.492857
two_tower_v2,0.204762,0.185714,68.671429,68.714286,0.473810
routed,0.466667,0.495238,77.624048,77.714286,0.735714
fusion_v1,0.495238,0.504762,69.683810,69.428571,0.590476
fusion_v2,0.157143,0.147619,71.721429,73.142857,0.552381


In [15]:
weight_results = []

for alpha, beta in [(0.7, 0.3), (0.5, 0.5), (0.3, 0.7)]:
    rows = []
    for _, q in queries.iterrows():
        qid = q["query_id"]
        strategy = q["strategy"]
        qdf = v1_top50[v1_top50["query_id"] == qid].copy()
        fused = fuse_query_df(qdf, strategy=strategy, alpha=alpha, beta=beta).head(50)
        rows.append(fused)

    fused_df = pd.concat(rows, ignore_index=True)
    mdf, sdf = evaluate_retrieval_df(fused_df, queries)
    s = summarize_metrics(mdf, sdf)
    s["alpha"] = alpha
    s["beta"] = beta
    weight_results.append(s)

pd.DataFrame(weight_results)

,top10_match_rate_mean,top20_match_rate_mean,top20_mean_score_mean,top20_median_score_mean,top20_high_score_rate_mean,alpha,beta
0,0.461905,0.478571,64.626429,65.747619,0.542857,0.7,0.3
1,0.495238,0.504762,69.683810,69.428571,0.590476,0.5,0.5
2,0.490476,0.514286,71.535714,71.204762,0.638095,0.3,0.7


In [16]:
def strategy_level_table(match_df: pd.DataFrame, score_df: pd.DataFrame) -> pd.DataFrame:
    a = match_df.groupby("strategy", as_index=False).agg(
        top10_match_rate_mean=("top10_match_rate", "mean"),
        top20_match_rate_mean=("top20_match_rate", "mean"),
    )
    b = score_df.groupby("strategy", as_index=False).agg(
        top20_mean_score_mean=("top20_mean_score", "mean"),
        top20_high_score_rate_mean=("top20_high_score_rate", "mean"),
    )
    return a.merge(b, on="strategy")

strategy_level_table(routed_match, routed_score)

,strategy,top10_match_rate_mean,top20_match_rate_mean,top20_mean_score_mean,top20_high_score_rate_mean
0,assembly_opportunity,0.000000,0.000000,83.400000,0.733333
1,dual_occupancy,1.000000,1.000000,72.433333,0.666667
2,granny_flat,0.000000,0.000000,74.333333,1.000000
3,land_bank_hold,0.666667,0.750000,71.235000,0.616667
4,low_rise_apartment,0.800000,0.900000,93.600000,0.900000
5,single_dwelling_rebuild,0.800000,0.816667,66.066667,0.433333
6,townhouse_multi_dwelling,0.000000,0.000000,82.300000,0.800000


In [17]:
best_top50 = routed_top50
best_match = routed_match
best_score = routed_score

out_dir = Path("../../artifacts/retrieval/hybrid_eval")
out_dir.mkdir(parents=True, exist_ok=True)

best_top50.to_parquet(out_dir / "best_top50.parquet", index=False)
best_match.to_parquet(out_dir / "best_match_metrics.parquet", index=False)
best_score.to_parquet(out_dir / "best_score_metrics.parquet", index=False)
summary_df.to_parquet(out_dir / "summary_table.parquet", index=False)

In [19]:
print(summary_df)

              top10_match_rate_mean  top20_match_rate_mean  \
heuristic                  0.385714               0.421429   
two_tower_v1               0.438095               0.457143   
two_tower_v2               0.204762               0.185714   
routed                     0.466667               0.495238   
fusion_v1                  0.495238               0.504762   
fusion_v2                  0.157143               0.147619   

              top20_mean_score_mean  top20_median_score_mean  \
heuristic                 88.714286                88.714286   
two_tower_v1              61.870000                62.116667   
two_tower_v2              68.671429                68.714286   
routed                    77.624048                77.714286   
fusion_v1                 69.683810                69.428571   
fusion_v2                 71.721429                73.142857   

              top20_high_score_rate_mean  
heuristic                       1.000000  
two_tower_v1                  

In [20]:
print(strategy_level_table(routed_match, routed_score))

                   strategy  top10_match_rate_mean  top20_match_rate_mean  \
0      assembly_opportunity               0.000000               0.000000   
1            dual_occupancy               1.000000               1.000000   
2               granny_flat               0.000000               0.000000   
3            land_bank_hold               0.666667               0.750000   
4        low_rise_apartment               0.800000               0.900000   
5   single_dwelling_rebuild               0.800000               0.816667   
6  townhouse_multi_dwelling               0.000000               0.000000   

   top20_mean_score_mean  top20_high_score_rate_mean  
0              83.400000                    0.733333  
1              72.433333                    0.666667  
2              74.333333                    1.000000  
3              71.235000                    0.616667  
4              93.600000                    0.900000  
5              66.066667                    0.433333 